In [ ]:
# ================================================================================================
# Purchase Prediction Pipeline — Final, Corrected Version
# ================================================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing   import OneHotEncoder, RobustScaler
from sklearn.impute          import SimpleImputer
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics         import (classification_report, roc_auc_score,
                                     f1_score, precision_recall_curve)
from scipy.sparse            import hstack
import xgboost as xgb


# ================================================================================================
# 0. Helper functions
# ================================================================================================

def undersample_train(X_train_fold, y_train_fold, pos_ratio=0.5, random_state=42):
    """
    Randomly undersample majority class so that the positive class makes up `pos_ratio` of the result.
    e.g. pos_ratio=0.5 gives a balanced 1:1 dataset.
    """
    pos_idx = y_train_fold[y_train_fold == 1].index
    neg_idx = y_train_fold[y_train_fold == 0].index
    n_pos = len(pos_idx)
    if n_pos == 0:
        return X_train_fold, y_train_fold
    # number of negatives to keep: n_pos * (1/pos_ratio - 1)
    n_neg = int(n_pos * (1.0 / pos_ratio - 1))
    n_neg = min(n_neg, len(neg_idx))
    rng = np.random.default_rng(random_state)
    sampled_neg = rng.choice(neg_idx, size=n_neg, replace=False)
    balanced_idx = np.concatenate([pos_idx, sampled_neg])
    rng.shuffle(balanced_idx)
    return X_train_fold.loc[balanced_idx], y_train_fold.loc[balanced_idx]


def tune_threshold(y_true, y_prob):
    """Find threshold that maximises F1 score."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    with np.errstate(divide='ignore', invalid='ignore'):
        f1_scores = 2 * (precision * recall) / (precision + recall)
        f1_scores = np.nan_to_num(f1_scores, nan=0.0)
    best_idx = np.argmax(f1_scores)
    best_thresh = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    return best_thresh, f1_scores[best_idx]


def get_feature_importance(model, model_name, num_cols, cat_cols, encoder):
    all_names = list(num_cols) + list(encoder.get_feature_names_out(cat_cols))
    scores = model.feature_importances_
    imp = pd.Series(scores, index=all_names).sort_values(ascending=False)
    return imp


# ================================================================================================
# 1. Load Data
# ================================================================================================
train = pd.read_csv('/Users/annageiser/Documents/GITHUB/analytics-project/data/raw/train.csv', sep='|')
items = pd.read_csv('/Users/annageiser/Documents/GITHUB/analytics-project/data/raw/items.csv', sep='|')
df = train.merge(items, on='pid', how='left', validate='m:1')
print("Data loaded and merged.")

# ================================================================================================
# 2. Sort and sample (use 0.5 for development; set to 1.0 for final run)
# ================================================================================================
df = df.sort_values(['day', 'lineID']).reset_index(drop=True)
df_sample = df.iloc[: int(len(df) * 0.1)].copy()
print(f"Using {len(df_sample):,} rows (10% sample).")

# ================================================================================================
# 3. Feature Engineering
# ================================================================================================
df_sample['priceRatio'] = (df_sample['price'] / df_sample['rrp'].replace(0, np.nan)).fillna(1.0)
df_sample['missingCompetitorPrice'] = df_sample['competitorPrice'].isnull().astype(int)

df_sample['weekDay_raw'] = (df_sample['day'] % 7).replace({0: 7})
df_sample['weekDay_sin'] = np.sin(2 * np.pi * df_sample['weekDay_raw'] / 7)
df_sample['weekDay_cos'] = np.cos(2 * np.pi * df_sample['weekDay_raw'] / 7)
df_sample.drop(columns=['weekDay_raw'], inplace=True)

df_sample['priceVsCompetitor'] = (
    df_sample['price'] / df_sample['competitorPrice'].replace(0, np.nan)
).fillna(1.0)
df_sample['priceDiscount'] = (df_sample['rrp'] - df_sample['price']).clip(lower=0)

df_sample['adFlag_x_priceRatio'] = df_sample['adFlag'] * df_sample['priceRatio']
df_sample['avail_x_priceRatio']  = df_sample['availability'] * df_sample['priceRatio']
df_sample['regulated_generic']   = df_sample['salesIndex'] * df_sample['genericProduct']

# ================================================================================================
# 4. Train / Test Split (time‑based, 80/20)
# ================================================================================================
X = df_sample.drop(columns=['order'])
y = df_sample['order']
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx].copy(), X.iloc[split_idx:].copy()
y_train, y_test = y.iloc[:split_idx].copy(), y.iloc[split_idx:].copy()
print(f"Train size: {len(X_train):,}  |  Test size: {len(X_test):,}")
print(f"Train positive rate: {y_train.mean():.3f}  |  Test positive rate: {y_test.mean():.3f}")

# ================================================================================================
# 5. Feature lists
# ================================================================================================
NUM_FEATURES = [
    'competitorPrice', 'priceRatio', 'priceVsCompetitor', 'priceDiscount',
    'missingCompetitorPrice', 'weekDay_sin', 'weekDay_cos', 'pid_order_rate',
    'pid_click_rate', 'pid_basket_rate', 'manufacturer_te', 'group_te',
    'category_te', 'adFlag_x_priceRatio', 'avail_x_priceRatio',
    'regulated_generic', 'pid_price_std', 'pid_seen_in_train'
]
CAT_FEATURES = [
    'adFlag', 'availability', 'content', 'unit', 'pharmForm',
    'genericProduct', 'salesIndex'
]
LEAKAGE_COLS = ['click', 'basket', 'revenue', 'price', 'rrp',
                'lineID', 'day', 'campaignIndex', 'pid']
TE_COLS = [
    ('manufacturer', 'manufacturer_te'),
    ('group',        'group_te'),
    ('category',     'category_te'),
]


# ================================================================================================
# 6. Cross‑validation experiment (rolling vs. expanding) – same as before, no changes needed
# ================================================================================================
# (The CV code remains unchanged. For brevity I omit it here; it is unchanged from the previous
#  complete script you received. You can keep the full CV block from the earlier answer.)
# ... (insert the full CV experiment from the previous complete answer) ...


# ================================================================================================
# 7. Final model — Corrected evaluation pipeline
# ================================================================================================
print("\n=== Final Model — Expanding‑Window Features + Correct Evaluation ===")

# ----- 7a. Build expanding‑window features for the whole training set (no leakage) -----
def build_expanding_train_features(df, y_series):
    """Same function as before, no changes."""
    df = df.copy()
    y_series = y_series.loc[df.index]
    df = df.sort_values('day').reset_index(drop=True)

    pid_total = {}; pid_order = {}; pid_click = {}; pid_basket = {}; pid_prices = {}
    grp_total = {}; grp_order = {}; man_total = {}; man_order = {}; cat_total = {}; cat_order = {}
    global_orders = 0; global_total = 0

    for col in ['pid_order_rate','pid_click_rate','pid_basket_rate','pid_price_std',
                'pid_seen_in_train','manufacturer_te','group_te','category_te']:
        df[col] = np.nan

    for day in sorted(df['day'].unique()):
        mask = df['day'] == day
        day_df = df.loc[mask]; y_day = y_series.loc[mask]
        prev_rate = global_orders / global_total if global_total > 0 else 0.0

        for pid_val in day_df['pid'].unique():
            idx = day_df.index[day_df['pid'] == pid_val]
            total_pid = pid_total.get(pid_val, 0)
            if total_pid > 0:
                df.loc[idx, 'pid_order_rate'] = pid_order.get(pid_val, 0) / total_pid
                df.loc[idx, 'pid_click_rate'] = pid_click.get(pid_val, 0) / total_pid
                df.loc[idx, 'pid_basket_rate'] = pid_basket.get(pid_val, 0) / total_pid
                df.loc[idx, 'pid_seen_in_train'] = 1
            else:
                df.loc[idx, ['pid_order_rate','pid_click_rate','pid_basket_rate']] = prev_rate
                df.loc[idx, 'pid_seen_in_train'] = 0
            past_prices = pid_prices.get(pid_val, [])
            df.loc[idx, 'pid_price_std'] = np.std(past_prices, ddof=0) if len(past_prices) >= 2 else 0.0

        for col_orig, col_te, order_dict, total_dict in [
            ('manufacturer','manufacturer_te', man_order, man_total),
            ('group','group_te', grp_order, grp_total),
            ('category','category_te', cat_order, cat_total)]:
            for val in day_df[col_orig].unique():
                idx_val = day_df.index[day_df[col_orig] == val]
                n_ord = order_dict.get(val, 0); n_tot = total_dict.get(val, 0)
                smoothing = 10
                te = ((n_ord + prev_rate*smoothing) / (n_tot + smoothing)) if (n_tot+smoothing) > 0 else prev_rate
                df.loc[idx_val, col_te] = te

        # update state
        pid_day_counts = day_df.groupby('pid').size()
        pid_day_orders = y_day.groupby(day_df['pid']).sum()
        for pid_val in pid_day_counts.index:
            pid_total[pid_val] = pid_total.get(pid_val,0) + pid_day_counts[pid_val]
            pid_order[pid_val] = pid_order.get(pid_val,0) + pid_day_orders.get(pid_val,0)
        for beh_col, cum_dict in [('click',pid_click),('basket',pid_basket)]:
            day_sum = day_df.groupby('pid')[beh_col].sum()
            for pid_val in day_sum.index:
                cum_dict[pid_val] = cum_dict.get(pid_val,0) + day_sum[pid_val]
        for pid_val, prices in day_df.groupby('pid')['price'].apply(list).items():
            if pid_val not in pid_prices: pid_prices[pid_val] = []
            pid_prices[pid_val].extend(prices)
        for col_orig, order_dict, total_dict in [
            ('manufacturer',man_order,man_total),
            ('group',grp_order,grp_total),
            ('category',cat_order,cat_total)]:
            day_grp = day_df.groupby(col_orig)
            day_cnt = day_grp.size(); day_ord = y_day.groupby(day_df[col_orig]).sum()
            for val in day_cnt.index:
                total_dict[val] = total_dict.get(val,0) + day_cnt[val]
                order_dict[val] = order_dict.get(val,0) + day_ord.get(val,0)
        global_total += len(day_df); global_orders += y_day.sum()
    return df

X_train_expanded = build_expanding_train_features(X_train.copy(), y_train)

# ----- 7b. Prepare a validation set (last 7 days of training) for threshold and HPO -----
unique_days = sorted(X_train_expanded['day'].unique())
val_frac = 0.1
val_size = max(1, int(len(unique_days) * val_frac))
val_days = unique_days[-val_size:]
train_train_days = [d for d in unique_days if d not in val_days]

print(f"Total days: {len(unique_days)} | train_train: {len(train_train_days)} | val: {len(val_days)}")

train_train_mask = X_train_expanded['day'].isin(train_train_days)
val_mask = X_train_expanded['day'].isin(val_days)

X_train_train = X_train_expanded[train_train_mask].copy()
y_train_train = y_train[train_train_mask].copy()
X_val = X_train_expanded[val_mask].copy()
y_val = y_train[val_mask].copy()

# Build test features using the full training state (expanding features already include all days)
# For test set, we use the final cumulative state from the last training day
def build_test_features(test_df, train_df_with_features):
    last_day = train_df_with_features['day'].max()
    last_train = train_df_with_features[train_df_with_features['day'] == last_day]
    pid_feats = last_train.groupby('pid')[['pid_order_rate','pid_click_rate','pid_basket_rate',
                                          'pid_price_std','pid_seen_in_train']].last()
    global_final_rate = last_train['pid_order_rate'].mean()
    test_df = test_df.join(pid_feats, on='pid')
    for col in ['pid_order_rate','pid_click_rate','pid_basket_rate','pid_price_std']:
        test_df[col] = test_df[col].fillna(global_final_rate if col != 'pid_price_std' else 0.0)
    test_df['pid_seen_in_train'] = test_df['pid_seen_in_train'].fillna(0).astype(int)
    for col_orig, col_te in TE_COLS:
        te_map = last_train.groupby(col_orig)[col_te].last()
        test_df[col_te] = test_df[col_orig].map(te_map).fillna(global_final_rate)
    return test_df

X_test_expanded = build_test_features(X_test.copy(), X_train_expanded)

# Drop TE original columns and leakage columns
for col, _ in TE_COLS:
    for df_part in [X_train_train, X_val, X_test_expanded]:
        df_part.drop(columns=[col], errors='ignore', inplace=True)
for df_part in [X_train_train, X_val, X_test_expanded]:
    df_part.drop(columns=LEAKAGE_COLS, errors='ignore', inplace=True)

# ----- 7c. Preprocessing (fit only on train_train) -----
# Correlation filter
active_num = [f for f in NUM_FEATURES if f in X_train_train.columns]
corr = X_train_train[active_num].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop = [col for col in upper.columns if any(upper[col] > 0.95)]
if drop: print(f"Dropping correlated features: {drop}")
for df_part in [X_train_train, X_val, X_test_expanded]:
    df_part.drop(columns=drop, errors='ignore', inplace=True)
active_num = [f for f in active_num if f not in drop]
active_cat = [f for f in CAT_FEATURES if f in X_train_train.columns]

# Impute & cast
fill = {col: 'Missing' for col in active_cat}
for df_part in [X_train_train, X_val, X_test_expanded]:
    df_part.fillna(fill, inplace=True)
    df_part[active_cat] = df_part[active_cat].astype(str)

# Encode & scale (fit on train_train ONLY)
imputer = SimpleImputer(strategy='median')
scaler  = RobustScaler()
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)

# ----- 7d. Undersample train_train (mild, pos_ratio=0.5) -----
X_trn_bal_df, y_trn_bal = undersample_train(X_train_train, y_train_train, pos_ratio=0.5)

X_trn_bal_num = scaler.fit_transform(
    imputer.fit_transform(X_trn_bal_df[active_num])
)

X_trn_bal_cat = encoder.fit_transform(
    X_trn_bal_df[active_cat]
)

X_trn_bal = hstack([X_trn_bal_num, X_trn_bal_cat])



X_val_num = scaler.transform(
    imputer.transform(X_val[active_num])
)

X_val_cat = encoder.transform(
    X_val[active_cat]
)

X_val_processed = hstack([X_val_num, X_val_cat])


X_tst_num = scaler.transform(
    imputer.transform(X_test_expanded[active_num])
)

X_tst_cat = encoder.transform(
    X_test_expanded[active_cat]
)

X_tst_processed = hstack([X_tst_num, X_tst_cat])



# ----- 7e. Hyperparameter tuning on train_train (XGBoost only for demo) -----
n_splits = min(3, len(train_train_days) - 1)
tscv = TimeSeriesSplit(n_splits=n_splits)
xgb_base = xgb.XGBClassifier(eval_metric='auc', use_label_encoder=False, random_state=42, verbosity=0)
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.7, 0.8],
    'min_child_weight': [3, 5],
}
search = RandomizedSearchCV(
    xgb_base, param_grid, cv=tscv, scoring='roc_auc',
    n_iter=10, random_state=42, n_jobs=-1
)
search.fit(X_trn_bal, y_trn_bal)
best_model = search.best_estimator_
print("Best XGBoost hyperparameters:", search.best_params_)

# ----- 7f. Find optimal threshold on validation set -----
y_val_prob = best_model.predict_proba(X_val_processed)[:, 1]
thresh, _ = tune_threshold(y_val, y_val_prob)
print(f"Optimal threshold on validation set: {thresh:.3f}")

# ----- 7g. Retrain best model on full training set (train_train + val, then undersample) -----
full_train = pd.concat([X_train_train, X_val])
full_y = pd.concat([y_train_train, y_val])
full_train_bal, full_y_bal = undersample_train(full_train, full_y, pos_ratio=0.5)

# Re-fit preprocessor on full training (or reuse, safe to refit)
# For simplicity, we'll refit scaler/imputer/encoder on full_train_bal
imputer_f = SimpleImputer(strategy='median')
scaler_f  = RobustScaler()
encoder_f = OneHotEncoder(handle_unknown='ignore', sparse_output=True)

# Preprocess full balanced training
X_full_num = scaler_f.fit_transform(imputer_f.fit_transform(full_train_bal[active_num]))
X_full_cat = encoder_f.fit_transform(full_train_bal[active_cat])
X_full = hstack([X_full_num, X_full_cat])

# Preprocess test set with the new preprocessors
X_tst_num = scaler_f.transform(imputer_f.transform(X_test_expanded[active_num]))
X_tst_cat = encoder_f.transform(X_test_expanded[active_cat])
X_tst_final = hstack([X_tst_num, X_tst_cat])

# Train final model with best hyperparameters
final_model = xgb.XGBClassifier(**search.best_params_, eval_metric='auc',
                                use_label_encoder=False, random_state=42, verbosity=0)
final_model.fit(X_full, full_y_bal)

# ----- 7h. Final evaluation on test set -----
y_test_prob = final_model.predict_proba(X_tst_final)[:, 1]
auc_test    = roc_auc_score(y_test, y_test_prob)          # Correct: using probabilities
y_test_pred = (y_test_prob >= thresh).astype(int)
f1_test     = f1_score(y_test, y_test_pred, zero_division=0)

print(f"Final XGBoost  |  AUC {auc_test:.4f}  |  F1 {f1_test:.4f}  |  Threshold {thresh:.3f}")
print(classification_report(y_test, y_test_pred, target_names=['no order', 'order']))

# Feature importance
imp = get_feature_importance(final_model, 'XGBoost', active_num, active_cat, encoder_f)
print("\nTop 15 features (XGBoost):")
print(imp.head(15).to_string())

pd.DataFrame({'metric': ['AUC','F1'], 'value': [auc_test, f1_test]}).to_csv('final_test_results.csv', index=False)
print("Final results saved.")

Data loaded and merged.
Using 275,600 rows (10% sample).
Train size: 220,480  |  Test size: 55,120
Train positive rate: 0.368  |  Test positive rate: 0.372

=== Final Model — Expanding‑Window Features + Correct Evaluation ===


In [6]:
print(y_train_train.value_counts())
print(y_val.value_counts())

Series([], Name: count, dtype: int64)
order
0    139342
1     81138
Name: count, dtype: int64
